# Тестовое задание TopFace Media: расчёт коэффициентов пролонгации

Цель notebook — подготовить данные, рассчитать коэффициенты пролонгации за 2023 год и выгрузить таблицы для отчёта в Google Sheets.

**Основные бизнес-правила из задания:**
- `month` в `prolongations.csv` — последний месяц реализации проекта.
- `AM` из `prolongations.csv` является первичным источником данных об аккаунт-менеджере относительно `Account` из `financial_data.csv`.
- Коэффициент 1-го месяца: отгрузка в первый месяц после завершения / отгрузка в последний месяц реализации.
- Коэффициент 2-го месяца: для проектов, не пролонгированных в 1-й месяц, отгрузка во второй месяц / отгрузка последнего месяца реализации.
- `стоп` и `end`: если встречаются в последний месяц реализации или ранее, соответствующая запись `prolongations` исключается из расчёта.
- `в ноль`: если отгрузка в месяце равна нулю, для расчёта используется отгрузка предыдущего месяца.

## 1. Загрузка библиотек и исходных данных

In [1]:
import re
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

In [2]:
# Загружаем исходные данные.
# Файлы должны находиться в той же папке, что и notebook.

prolongations_raw = pd.read_csv('prolongations.csv')
financial_data_raw = pd.read_csv('financial_data.csv')

print('prolongations_raw:', prolongations_raw.shape)
print('financial_data_raw:', financial_data_raw.shape)

prolongations_raw: (477, 3)
financial_data_raw: (451, 19)


In [3]:
# Быстрый визуальный контроль структуры данных.

display(prolongations_raw.head())
display(financial_data_raw.head())

,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич
3,87,ноябрь 2022,Соколова Анастасия Викторовна
4,429,ноябрь 2022,Соколова Анастасия Викторовна


,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
3,594,NaN,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
4,665,NaN,"10 000,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


## 2. Первичная проверка качества данных

In [4]:
# Проверяем типы данных и количество непустых значений.
# В financial_data месячные колонки имеют тип object,
# так как в них смешаны числа, пропуски и специальные текстовые значения.

print('Prolongations info:')
prolongations_raw.info()

print('\nFinancial data info:')
financial_data_raw.info()

Prolongations info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477 entries, 0 to 476
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      477 non-null    int64 
 1   month   477 non-null    object
 2   AM      477 non-null    object
dtypes: int64(1), object(2)
memory usage: 11.3+ KB

Financial data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 451 entries, 0 to 450
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             451 non-null    int64 
 1   Причина дубля  301 non-null    object
 2   Ноябрь 2022    156 non-null    object
 3   Декабрь 2022   159 non-null    object
 4   Январь 2023    139 non-null    object
 5   Февраль 2023   145 non-null    object
 6   Март 2023      168 non-null    object
 7   Апрель 2023    174 non-null    object
 8   Май 2023       190 non-null    object
 9   Июнь 2023      190 non-null    object
 1

In [5]:
# Проверяем значения в столбце "Причина дубля".
# В дальнейшем строки с несколькими финансовыми частями будут агрегироваться
# на уровне project id + месяц.

display(financial_data_raw['Причина дубля'].value_counts(dropna=False))

Причина дубля
NaN                    150
первая часть оплаты    114
вторая часть оплаты     99
доп работы              38
основные работы         38
изменение ЮЛ            11
карты, банки             1
Name: count, dtype: int64

In [6]:
# Проверяем покрытие project id между таблицами.
# Важно убедиться, что проекты из prolongations присутствуют в финансовых данных.

prolongations_ids = set(prolongations_raw['id'])
financial_data_ids = set(financial_data_raw['id'])

only_in_prolongations = prolongations_ids - financial_data_ids
only_in_financial_data = financial_data_ids - prolongations_ids

print('id только в prolongations:', len(only_in_prolongations), only_in_prolongations)
print('id только в financial_data:', len(only_in_financial_data), only_in_financial_data)

# Проект 930 присутствует только в financial_data и не используется в расчётах,
# так как расчётная база строится от prolongations.
if only_in_financial_data:
    display(financial_data_raw[financial_data_raw['id'].isin(only_in_financial_data)])

id только в prolongations: 0 set()
id только в financial_data: 1 {930}


,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
450,930,первая часть оплаты,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"36 687,50","31 152,50","30 350,00","34 547,50","25 750,00","25 600,00",Кузнецов Михаил Иванович


In [7]:
# Проверяем справочники менеджеров в обеих таблицах.
# Списки совпадают, но для расчётов используется AM из prolongations,
# так как он является первичным источником по условию задачи.

prolongations_managers = set(prolongations_raw['AM'].unique())
financial_managers = set(financial_data_raw['Account'].unique())

print('Менеджеров в prolongations:', len(prolongations_managers))
print('Менеджеров в financial_data:', len(financial_managers))
print('Только в prolongations:', prolongations_managers - financial_managers)
print('Только в financial_data:', financial_managers - prolongations_managers)

print('\nСписок менеджеров:')
for manager in sorted(prolongations_managers):
    print('-', manager)

Менеджеров в prolongations: 10
Менеджеров в financial_data: 10
Только в prolongations: set()
Только в financial_data: set()

Список менеджеров:
- Васильев Артем Александрович
- Иванова Мария Сергеевна
- Кузнецов Михаил Иванович
- Михайлов Андрей Сергеевич
- Петрова Анна Дмитриевна
- Попова Екатерина Николаевна
- Смирнова Ольга Владимировна
- Соколова Анастасия Викторовна
- Федорова Марина Васильевна
- без А/М


In [8]:
# Проверяем записи без аккаунт-менеджера.
# Проект 907 не содержит финансовых данных и далее будет исключён.
# Категория "без А/М" сохраняется как отдельная категория, так как проект 1004 имеет финансовые данные.

projects_without_am = sorted(
    set(prolongations_raw.loc[prolongations_raw['AM'] == 'без А/М', 'id'])
    | set(financial_data_raw.loc[financial_data_raw['Account'] == 'без А/М', 'id'])
)

print('Проекты без А/М:', projects_without_am)

display(prolongations_raw[prolongations_raw['id'].isin(projects_without_am)].sort_values('id'))
display(financial_data_raw[financial_data_raw['id'].isin(projects_without_am)].sort_values('id'))

Проекты без А/М: [907, 1004]


,id,month,AM
273,907,июль 2023,без А/М
473,1004,декабрь 2023,без А/М


,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
195,907,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,без А/М
445,1004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"34 000,00","0,00",NaN,NaN,без А/М


In [9]:
# Проверяем формат месяца в prolongations.
# Все значения должны иметь формат "месяц + год".

display(prolongations_raw['month'].value_counts().sort_index())

month
август 2023      26
апрель 2023      28
декабрь 2022     67
декабрь 2023     69
июль 2023        23
июнь 2023        31
май 2023         22
март 2023        33
ноябрь 2022      24
ноябрь 2023      40
октябрь 2023     24
сентябрь 2023    42
февраль 2023     28
январь 2023      20
Name: count, dtype: int64

In [10]:
# Выделяем месячные колонки financial_data.
# По условию в них находятся суммы отгрузки и специальные значения.

month_columns_from_fd = financial_data_raw.columns[2:-1]

print('Месячные колонки financial_data:')
for col in month_columns_from_fd:
    print('-', col)

Месячные колонки financial_data:
- Ноябрь 2022
- Декабрь 2022
- Январь 2023
- Февраль 2023
- Март 2023
- Апрель 2023
- Май 2023
- Июнь 2023
- Июль 2023
- Август 2023
- Сентябрь 2023
- Октябрь 2023
- Ноябрь 2023
- Декабрь 2023
- Январь 2024
- Февраль 2024


In [11]:
# Проверяем известные специальные значения из условия задачи.

known_special_values = ['в ноль', 'стоп', 'end']

for value in known_special_values:
    count = financial_data_raw[month_columns_from_fd].isin([value]).sum().sum()
    print(f'{value}: {count}')

в ноль: 129
стоп: 72
end: 1


In [12]:
# Проверяем, есть ли в месячных колонках неожиданные нечисловые значения,
# кроме известных специальных значений: "в ноль", "стоп", "end".

unexpected_values = set()

for col in month_columns_from_fd:
    for value in financial_data_raw[col].dropna().unique():
        value_str = str(value).strip()

        if value_str in known_special_values:
            continue

        cleaned_value = re.sub(r'\s+', '', value_str).replace(',', '.')

        try:
            float(cleaned_value)
        except ValueError:
            unexpected_values.add(value_str)

print('Неожиданные значения:', sorted(unexpected_values))

Неожиданные значения: []


In [13]:
# Проверяем полные дубликаты в исходных таблицах.
# В prolongations есть точные дубликаты, которые будут удалены на этапе очистки.

print('Полные дубликаты в prolongations:', prolongations_raw.duplicated().sum())
print('Полные дубликаты в financial_data:', financial_data_raw.duplicated().sum())

display(
    prolongations_raw[
        prolongations_raw.duplicated(keep=False)
    ].sort_values(['id', 'month', 'AM'])
)

Полные дубликаты в prolongations: 3


Полные дубликаты в financial_data: 0


,id,month,AM
186,600,апрель 2023,Васильев Артем Александрович
187,600,апрель 2023,Васильев Артем Александрович
160,682,март 2023,Васильев Артем Александрович
161,682,март 2023,Васильев Артем Александрович
163,697,март 2023,Васильев Артем Александрович
164,697,март 2023,Васильев Артем Александрович


## 3. Подготовка `prolongations`

На этом этапе:
- преобразуем `month` в дату;
- удаляем точные дубликаты;
- исправляем конфликтный кейс по проекту `361`;
- исключаем записи, где проект завершился досрочно (`стоп/end`).

In [14]:
# Создаём рабочие копии исходных данных.
# Raw-таблицы не изменяем, чтобы сохранить возможность проверки и отката.

prolongations = prolongations_raw.copy()
financial_data = financial_data_raw.copy()

In [15]:
# Преобразуем русский текстовый месяц в datetime.
# Дата приводится к первому числу месяца.

month_map = {
    'январь': 'January',
    'февраль': 'February',
    'март': 'March',
    'апрель': 'April',
    'май': 'May',
    'июнь': 'June',
    'июль': 'July',
    'август': 'August',
    'сентябрь': 'September',
    'октябрь': 'October',
    'ноябрь': 'November',
    'декабрь': 'December'
}

def convert_russian_month_to_datetime(value):
    month_name, year = value.lower().split()
    english_month = month_map[month_name]
    return pd.to_datetime(f'{english_month} {year}', format='%B %Y')

prolongations['month_dt'] = prolongations['month'].apply(convert_russian_month_to_datetime)

display(
    prolongations[['month', 'month_dt']]
    .drop_duplicates()
    .sort_values('month_dt')
)

,month,month_dt
0,ноябрь 2022,2022-11-01
24,декабрь 2022,2022-12-01
91,январь 2023,2023-01-01
111,февраль 2023,2023-02-01
139,март 2023,2023-03-01
172,апрель 2023,2023-04-01
200,май 2023,2023-05-01
222,июнь 2023,2023-06-01
253,июль 2023,2023-07-01
276,август 2023,2023-08-01


In [16]:
# Удаляем точные дубликаты prolongations по id + month + AM.
# Финансовые части проекта агрегируются отдельно в financial_data,
# поэтому повторная строка в prolongations не должна задваивать попытку пролонгации.

print('Размер до удаления дублей:', prolongations.shape)

prolongations_duplicates = prolongations[
    prolongations.duplicated(subset=['id', 'month', 'AM'], keep=False)
].sort_values(['id', 'month', 'AM'])

display(prolongations_duplicates)

prolongations = (
    prolongations
    .drop_duplicates(subset=['id', 'month', 'AM'])
    .copy()
)

print('Размер после удаления дублей:', prolongations.shape)
print('Оставшиеся дубли:', prolongations.duplicated(subset=['id', 'month', 'AM']).sum())

Размер до удаления дублей: (477, 4)


,id,month,AM,month_dt
186,600,апрель 2023,Васильев Артем Александрович,2023-04-01
187,600,апрель 2023,Васильев Артем Александрович,2023-04-01
160,682,март 2023,Васильев Артем Александрович,2023-03-01
161,682,март 2023,Васильев Артем Александрович,2023-03-01
163,697,март 2023,Васильев Артем Александрович,2023-03-01
164,697,март 2023,Васильев Артем Александрович,2023-03-01


Размер после удаления дублей: (474, 4)
Оставшиеся дубли: 0


In [17]:
# Проверяем конфликты: один id + month, но несколько разных AM.
# Такой кейс был найден для проекта 361 в сентябре 2023.

am_conflicts = (
    prolongations
    .groupby(['id', 'month', 'month_dt'])['AM']
    .nunique()
    .reset_index(name='am_count')
    .query('am_count > 1')
)

display(am_conflicts)

conflict_rows = prolongations.merge(
    am_conflicts[['id', 'month', 'month_dt']],
    on=['id', 'month', 'month_dt'],
    how='inner'
).sort_values(['id', 'month_dt', 'AM'])

display(conflict_rows)

,id,month,month_dt,am_count
87,361,сентябрь 2023,2023-09-01,2


,id,month,AM,month_dt
1,361,сентябрь 2023,Попова Екатерина Николаевна,2023-09-01
0,361,сентябрь 2023,Смирнова Ольга Владимировна,2023-09-01


In [18]:
# Обработка конфликта id=361, сентябрь 2023.
# По предыдущим и последующим месяцам проект относится к Поповой Екатерине Николаевне.
# Запись с другим менеджером считаем ошибочной и исключаем.

conflict_mask = (
    (prolongations['id'] == 361)
    & (prolongations['month'] == 'сентябрь 2023')
    & (prolongations['AM'] != 'Попова Екатерина Николаевна')
)

print('Удаляемая конфликтная запись:')
display(prolongations[conflict_mask])

prolongations = prolongations[~conflict_mask].copy()

# Контроль: после исправления конфликтов быть не должно.
am_conflicts_after = (
    prolongations
    .groupby(['id', 'month', 'month_dt'])['AM']
    .nunique()
    .reset_index(name='am_count')
    .query('am_count > 1')
)

print('Конфликты после исправления:', len(am_conflicts_after))
display(am_conflicts_after)

Удаляемая конфликтная запись:


,id,month,AM,month_dt
324,361,сентябрь 2023,Смирнова Ольга Владимировна,2023-09-01


Конфликты после исправления:

 0


,id,month,month_dt,am_count


## 4. Подготовка `financial_data`

In [19]:
# Переводим financial_data из wide-формата в long-формат.
# Теперь одна строка = project id + месяц + исходное значение отгрузки.

financial_data_long = financial_data.melt(
    id_vars=['id', 'Причина дубля', 'Account'],
    value_vars=month_columns_from_fd,
    var_name='month',
    value_name='shipment_raw'
)

financial_data_long['month_dt'] = financial_data_long['month'].apply(convert_russian_month_to_datetime)

print('Размер financial_data:', financial_data.shape)
print('Размер financial_data_long:', financial_data_long.shape)

display(financial_data_long.head())

Размер financial_data: (451, 19)
Размер financial_data_long: (7216, 6)


,id,Причина дубля,Account,month,shipment_raw,month_dt
0,42,NaN,Васильев Артем Александрович,Ноябрь 2022,"36 220,00",2022-11-01
1,657,первая часть оплаты,Васильев Артем Александрович,Ноябрь 2022,стоп,2022-11-01
2,657,вторая часть оплаты,Васильев Артем Александрович,Ноябрь 2022,стоп,2022-11-01
3,594,NaN,Васильев Артем Александрович,Ноябрь 2022,стоп,2022-11-01
4,665,NaN,Васильев Артем Александрович,Ноябрь 2022,"10 000,00",2022-11-01


In [20]:
# Создаём флаги специальных значений и числовой столбец shipment.
# "стоп/end" и "в ноль" не переводятся в shipment как числа:
# для них предусмотрена отдельная бизнес-логика.

financial_data_long['shipment_raw_str'] = financial_data_long['shipment_raw'].astype(str).str.strip()

financial_data_long['is_zero_text'] = financial_data_long['shipment_raw_str'].eq('в ноль')
financial_data_long['is_stop'] = financial_data_long['shipment_raw_str'].isin(['стоп', 'end'])

def clean_shipment_value(value):
    if pd.isna(value):
        return np.nan

    value_str = str(value).strip()

    if value_str in ['в ноль', 'стоп', 'end']:
        return np.nan

    cleaned_value = re.sub(r'\s+', '', value_str).replace(',', '.')
    return float(cleaned_value)

financial_data_long['shipment'] = financial_data_long['shipment_raw'].apply(clean_shipment_value)

financial_data_long['is_numeric_zero'] = (
    financial_data_long['shipment'].eq(0)
    & financial_data_long['shipment_raw'].notna()
    & ~financial_data_long['is_zero_text']
    & ~financial_data_long['is_stop']
)

display(financial_data_long.head(20))

,id,Причина дубля,Account,month,shipment_raw,month_dt,shipment_raw_str,is_zero_text,is_stop,shipment,is_numeric_zero
0,42,NaN,Васильев Артем Александрович,Ноябрь 2022,"36 220,00",2022-11-01,"36 220,00",False,False,36220.0,False
1,657,первая часть оплаты,Васильев Артем Александрович,Ноябрь 2022,стоп,2022-11-01,стоп,False,True,NaN,False
2,657,вторая часть оплаты,Васильев Артем Александрович,Ноябрь 2022,стоп,2022-11-01,стоп,False,True,NaN,False
3,594,NaN,Васильев Артем Александрович,Ноябрь 2022,стоп,2022-11-01,стоп,False,True,NaN,False
4,665,NaN,Васильев Артем Александрович,Ноябрь 2022,"10 000,00",2022-11-01,"10 000,00",False,False,10000.0,False
5,637,NaN,Соколова Анастасия Викторовна,Ноябрь 2022,"38 045,00",2022-11-01,"38 045,00",False,False,38045.0,False
6,419,NaN,Михайлов Андрей Сергеевич,Ноябрь 2022,NaN,2022-11-01,nan,False,False,NaN,False
7,578,NaN,Попова Екатерина Николаевна,Ноябрь 2022,"82 800,00",2022-11-01,"82 800,00",False,False,82800.0,False
8,579,NaN,Васильев Артем Александрович,Ноябрь 2022,в ноль,2022-11-01,в ноль,True,False,NaN,False
9,592,первая часть оплаты,Васильев Артем Александрович,Ноябрь 2022,"92 302,00",2022-11-01,"92 302,00",False,False,92302.0,False


In [21]:
# Контрольная проверка: все непустые значения должны быть распознаны
# либо как число, либо как специальное значение.

unprocessed_values = financial_data_long[
    financial_data_long['shipment_raw'].notna()
    & financial_data_long['shipment'].isna()
    & ~financial_data_long['is_zero_text']
    & ~financial_data_long['is_stop']
]

print('Количество "в ноль":', financial_data_long['is_zero_text'].sum())
print('Количество "стоп/end":', financial_data_long['is_stop'].sum())
print('Количество числовых значений shipment:', financial_data_long['shipment'].notna().sum())
print('Необработанные значения:', len(unprocessed_values))

display(unprocessed_values)

Количество "в ноль": 129
Количество "стоп/end": 73
Количество числовых значений shipment: 2394
Необработанные значения: 0


,id,Причина дубля,Account,month,shipment_raw,month_dt,shipment_raw_str,is_zero_text,is_stop,shipment,is_numeric_zero


## 5. Исключение проектов со `стоп/end`

In [22]:
# События stop/end означают досрочное завершение проекта.
# Если stop/end был в последний месяц реализации или раньше,
# соответствующая запись prolongations исключается из расчёта.

stop_events = (
    financial_data_long
    .loc[financial_data_long['is_stop'], ['id', 'month_dt']]
    .drop_duplicates()
)

print('Количество stop/end событий:', len(stop_events))
print('Количество проектов со stop/end:', stop_events['id'].nunique())

display(stop_events.head())

Количество stop/end событий: 57
Количество проектов со stop/end: 55


,id,month_dt
1,657,2022-11-01
3,594,2022-11-01
64,626,2022-11-01
470,572,2022-12-01
471,573,2022-12-01


In [23]:
# Присоединяем stop/end события к prolongations по id
# и проверяем условие month_dt_stop <= month_dt.

prolongations_stop_check = prolongations.merge(
    stop_events,
    on='id',
    how='left',
    suffixes=('', '_stop')
)

prolongations_stop_check['exclude_by_stop'] = (
    prolongations_stop_check['month_dt_stop'].notna()
    & (prolongations_stop_check['month_dt_stop'] <= prolongations_stop_check['month_dt'])
)

prolongations_to_exclude_by_stop = (
    prolongations_stop_check
    .loc[prolongations_stop_check['exclude_by_stop'], ['id', 'month', 'month_dt', 'AM']]
    .drop_duplicates()
)

print('Записей prolongations к исключению по stop/end:', len(prolongations_to_exclude_by_stop))
print('Уникальных проектов к исключению:', prolongations_to_exclude_by_stop['id'].nunique())

display(prolongations_to_exclude_by_stop.head(20))

Записей prolongations к исключению по stop/end: 48
Уникальных проектов к исключению: 46


,id,month,month_dt,AM
6,657,ноябрь 2022,2022-11-01,Васильев Артем Александрович
7,594,ноябрь 2022,2022-11-01,Васильев Артем Александрович
28,572,декабрь 2022,2022-12-01,Васильев Артем Александрович
29,573,декабрь 2022,2022-12-01,Васильев Артем Александрович
30,574,декабрь 2022,2022-12-01,Васильев Артем Александрович
31,575,декабрь 2022,2022-12-01,Васильев Артем Александрович
32,576,декабрь 2022,2022-12-01,Васильев Артем Александрович
41,613,декабрь 2022,2022-12-01,Васильев Артем Александрович
42,615,декабрь 2022,2022-12-01,Васильев Артем Александрович
43,617,декабрь 2022,2022-12-01,Васильев Артем Александрович


In [24]:
# Создаём prolongations_clean:
# исключаем только конкретные записи id + month + AM,
# а не весь проект целиком.

prolongations_clean = prolongations.merge(
    prolongations_to_exclude_by_stop.assign(exclude_by_stop=True),
    on=['id', 'month', 'month_dt', 'AM'],
    how='left'
)

prolongations_clean = (
    prolongations_clean
    .loc[prolongations_clean['exclude_by_stop'].isna()]
    .drop(columns='exclude_by_stop')
    .copy()
)

print('Размер prolongations до stop/end:', prolongations.shape)
print('Размер prolongations_clean:', prolongations_clean.shape)

Размер prolongations до stop/end: (473, 4)
Размер prolongations_clean: (425, 4)


## 6. Расчёт отгрузки для использования в коэффициентах

In [25]:
# Агрегируем financial_data на уровне project id + месяц.
# Важно: sum(min_count=1) не превращает полностью пустой месяц в 0.
# Это позволяет отличить отсутствие данных от реальной нулевой отгрузки.

zero_logic_check = (
    financial_data_long
    .groupby(['id', 'month_dt'])
    .agg(
        shipment_sum=('shipment', lambda x: x.sum(min_count=1)),
        total_rows=('shipment_raw', 'size'),
        non_null_rows=('shipment_raw', lambda x: x.notna().sum()),
        zero_text_count=('is_zero_text', 'sum'),
        numeric_zero_count=('is_numeric_zero', 'sum'),
        stop_count=('is_stop', 'sum'),
        positive_shipment_count=('shipment', lambda x: (x > 0).sum())
    )
    .reset_index()
)

zero_logic_check['zero_markers_count'] = (
    zero_logic_check['zero_text_count']
    + zero_logic_check['numeric_zero_count']
)

# Месяц считается нулевым, если есть хотя бы один признак нулевой отгрузки
# ("в ноль" или числовой 0), нет положительной отгрузки и нет stop/end.
zero_logic_check['is_full_zero_month'] = (
    (zero_logic_check['zero_markers_count'] > 0)
    & (zero_logic_check['positive_shipment_count'] == 0)
    & (zero_logic_check['stop_count'] == 0)
)

display(zero_logic_check.head(20))

,id,month_dt,shipment_sum,total_rows,non_null_rows,zero_text_count,numeric_zero_count,stop_count,positive_shipment_count,zero_markers_count,is_full_zero_month
0,15,2022-11-01,439280.00,2,2,0,0,0,2,0,False
1,15,2022-12-01,439280.00,2,2,0,0,0,2,0,False
2,15,2023-01-01,102433.75,2,1,0,0,0,1,0,False
3,15,2023-02-01,102433.75,2,1,0,0,0,1,0,False
4,15,2023-03-01,102433.75,2,1,0,0,0,1,0,False
5,15,2023-04-01,138158.00,2,1,0,0,0,1,0,False
6,15,2023-05-01,138158.00,2,1,0,0,0,1,0,False
7,15,2023-06-01,102433.75,2,1,0,0,0,1,0,False
8,15,2023-07-01,NaN,2,0,0,0,0,0,0,False
9,15,2023-08-01,NaN,2,0,0,0,0,0,0,False


In [26]:
# Для месяцев "в ноль" используем отгрузку предыдущего месяца.
# Расчёт выполняется последовательно внутри каждого проекта:
# если предыдущую отгрузку восстановить невозможно, оставляем NaN.

zero_logic_check = zero_logic_check.sort_values(['id', 'month_dt']).copy()

def calculate_shipment_for_project(group):
    group = group.sort_values('month_dt').copy()
    calculated_values = []

    for _, row in group.iterrows():
        if row['is_full_zero_month']:
            if calculated_values:
                calculated_values.append(calculated_values[-1])
            else:
                calculated_values.append(np.nan)
        else:
            calculated_values.append(row['shipment_sum'])

    group['shipment_for_calc'] = calculated_values
    return group

processed_groups = []

for project_id, group in zero_logic_check.groupby('id', sort=False):
    processed_groups.append(calculate_shipment_for_project(group))

zero_logic_check = pd.concat(processed_groups, ignore_index=True)

display(
    zero_logic_check[
        zero_logic_check['is_full_zero_month']
    ][['id', 'month_dt', 'shipment_sum', 'shipment_for_calc']].head(20)
)

,id,month_dt,shipment_sum,shipment_for_calc
421,211,2023-04-01,NaN,NaN
864,453,2022-11-01,NaN,NaN
1456,579,2022-11-01,NaN,NaN
1479,580,2023-06-01,NaN,57901.14
1481,580,2023-08-01,NaN,64219.76
1664,613,2022-11-01,NaN,NaN
1680,615,2022-11-01,NaN,NaN
1696,617,2022-11-01,NaN,NaN
1712,620,2022-11-01,NaN,NaN
1856,643,2022-11-01,NaN,NaN


In [27]:
# Формируем таблицу отгрузок для расчётов.
# Проект 907 исключается: по нему нет финансовых данных и нет закреплённого AM.

shipments_for_calc = zero_logic_check[
    ['id', 'month_dt', 'shipment_for_calc']
].copy()

shipments_for_calc = shipments_for_calc[
    shipments_for_calc['id'] != 907
].copy()

print('Размер shipments_for_calc:', shipments_for_calc.shape)
print('NaN в shipment_for_calc:', shipments_for_calc['shipment_for_calc'].isna().sum())

display(shipments_for_calc.head())

Размер shipments_for_calc: (5008, 3)
NaN в shipment_for_calc: 3178


,id,month_dt,shipment_for_calc
0,15,2022-11-01,439280.00
1,15,2022-12-01,439280.00
2,15,2023-01-01,102433.75
3,15,2023-02-01,102433.75
4,15,2023-03-01,102433.75


In [28]:
# Исключаем проект 907 из prolongations_clean.
# Категория "без А/М" сохраняется как отдельная категория, если по проекту есть финансовые данные.

prolongations_clean = prolongations_clean[
    prolongations_clean['id'] != 907
].copy()

print('907 в prolongations_clean:', (prolongations_clean['id'] == 907).sum())
print('907 в shipments_for_calc:', (shipments_for_calc['id'] == 907).sum())

907 в prolongations_clean: 0
907 в shipments_for_calc: 0


## 7. Формирование расчётной базы пролонгаций

In [29]:
# Каждая строка prolongations_clean — отдельная попытка/период пролонгации.
# AM берём из prolongations, так как это первичный источник по условию задачи.
# Для каждой попытки добавляем даты первого и второго месяца после завершения.

prolongation_base = prolongations_clean.copy()

prolongation_base['month_1_dt'] = prolongation_base['month_dt'] + pd.DateOffset(months=1)
prolongation_base['month_2_dt'] = prolongation_base['month_dt'] + pd.DateOffset(months=2)

display(prolongation_base.head())

,id,month,AM,month_dt,month_1_dt,month_2_dt
0,42,ноябрь 2022,Васильев Артем Александрович,2022-11-01,2022-12-01,2023-01-01
1,453,ноябрь 2022,Васильев Артем Александрович,2022-11-01,2022-12-01,2023-01-01
2,548,ноябрь 2022,Михайлов Андрей Сергеевич,2022-11-01,2022-12-01,2023-01-01
3,87,ноябрь 2022,Соколова Анастасия Викторовна,2022-11-01,2022-12-01,2023-01-01
4,429,ноябрь 2022,Соколова Анастасия Викторовна,2022-11-01,2022-12-01,2023-01-01


In [30]:
# Подтягиваем отгрузку последнего месяца реализации проекта.
# Это знаменатель для коэффициента 1-го месяца.

end_month_shipments = shipments_for_calc.rename(columns={
    'shipment_for_calc': 'end_month_shipment'
})

prolongation_base = prolongation_base.merge(
    end_month_shipments,
    on=['id', 'month_dt'],
    how='left'
)

# Подтягиваем отгрузку M+1.

month_1_shipments = shipments_for_calc.rename(columns={
    'month_dt': 'month_1_dt',
    'shipment_for_calc': 'month_1_shipment'
})

prolongation_base = prolongation_base.merge(
    month_1_shipments,
    on=['id', 'month_1_dt'],
    how='left'
)

# Подтягиваем отгрузку M+2.

month_2_shipments = shipments_for_calc.rename(columns={
    'month_dt': 'month_2_dt',
    'shipment_for_calc': 'month_2_shipment'
})

prolongation_base = prolongation_base.merge(
    month_2_shipments,
    on=['id', 'month_2_dt'],
    how='left'
)

# Если в M+1 или M+2 нет финансовых данных, считаем, что отгрузки не было.
# Для end_month_shipment NaN не заполняем: без него нельзя посчитать знаменатель.
prolongation_base['month_1_shipment'] = prolongation_base['month_1_shipment'].fillna(0)
prolongation_base['month_2_shipment'] = prolongation_base['month_2_shipment'].fillna(0)

display(prolongation_base.head())

,id,month,AM,month_dt,month_1_dt,month_2_dt,end_month_shipment,month_1_shipment,month_2_shipment
0,42,ноябрь 2022,Васильев Артем Александрович,2022-11-01,2022-12-01,2023-01-01,36220.0,0.0,0.0
1,453,ноябрь 2022,Васильев Артем Александрович,2022-11-01,2022-12-01,2023-01-01,NaN,39245.0,44320.0
2,548,ноябрь 2022,Михайлов Андрей Сергеевич,2022-11-01,2022-12-01,2023-01-01,674000.0,674000.0,674000.0
3,87,ноябрь 2022,Соколова Анастасия Викторовна,2022-11-01,2022-12-01,2023-01-01,70050.0,0.0,73380.0
4,429,ноябрь 2022,Соколова Анастасия Викторовна,2022-11-01,2022-12-01,2023-01-01,30280.0,35580.0,35830.0


In [31]:
# Исключаем записи, где невозможно определить отгрузку последнего месяца реализации.
# Это может быть связано с отсутствием предыдущего месяца для восстановления "в ноль"
# или полным отсутствием финансовых данных.

missing_end_month = prolongation_base[
    prolongation_base['end_month_shipment'].isna()
]

print('Записей без end_month_shipment:', len(missing_end_month))
display(missing_end_month)

prolongation_base = prolongation_base[
    prolongation_base['end_month_shipment'].notna()
].copy()

# Дополнительно исключаем строки с нулевым знаменателем.
# Они не должны участвовать в коэффициентах.
prolongation_base = prolongation_base[
    prolongation_base['end_month_shipment'] > 0
].copy()

print('Итоговый размер prolongation_base:', prolongation_base.shape)

Записей без end_month_shipment: 26


,id,month,AM,month_dt,month_1_dt,month_2_dt,end_month_shipment,month_1_shipment,month_2_shipment
1,453,ноябрь 2022,Васильев Артем Александрович,2022-11-01,2022-12-01,2023-01-01,NaN,39245.0,44320.0
9,419,ноябрь 2022,Михайлов Андрей Сергеевич,2022-11-01,2022-12-01,2023-01-01,NaN,0.0,0.0
11,579,ноябрь 2022,Васильев Артем Александрович,2022-11-01,2022-12-01,2023-01-01,NaN,0.0,0.0
19,620,ноябрь 2022,Васильев Артем Александрович,2022-11-01,2022-12-01,2023-01-01,NaN,0.0,0.0
23,16,декабрь 2022,Иванова Мария Сергеевна,2022-12-01,2023-01-01,2023-02-01,NaN,0.0,0.0
38,603,декабрь 2022,Михайлов Андрей Сергеевич,2022-12-01,2023-01-01,2023-02-01,NaN,0.0,0.0
66,714,декабрь 2022,Попова Екатерина Николаевна,2022-12-01,2023-01-01,2023-02-01,NaN,0.0,0.0
75,684,январь 2023,Васильев Артем Александрович,2023-01-01,2023-02-01,2023-03-01,NaN,0.0,0.0
91,230,январь 2023,Соколова Анастасия Викторовна,2023-01-01,2023-02-01,2023-03-01,NaN,0.0,0.0
94,732,январь 2023,Кузнецов Михаил Иванович,2023-01-01,2023-02-01,2023-03-01,NaN,0.0,0.0


Итоговый размер prolongation_base: (398, 9)


In [32]:
# Финальная проверка расчётной базы перед расчётом коэффициентов.

duplicates_in_prolongations = prolongations.duplicated(
    subset=['id', 'month', 'AM']
).sum()

am_conflicts_final = (
    prolongations
    .groupby(['id', 'month', 'month_dt'])['AM']
    .nunique()
    .reset_index(name='am_count')
    .query('am_count > 1')
)

base_duplicates = prolongation_base.duplicated(
    subset=['id', 'month', 'AM']
).sum()

base_am_conflicts = (
    prolongation_base
    .groupby(['id', 'month', 'month_dt'])['AM']
    .nunique()
    .reset_index(name='am_count')
    .query('am_count > 1')
)

checks = {
    'Нет дублей в prolongations': duplicates_in_prolongations == 0,
    'Нет конфликтов AM в prolongations': len(am_conflicts_final) == 0,
    'Проект 907 исключён из prolongations_clean': (prolongations_clean['id'] == 907).sum() == 0,
    'Проект 907 исключён из shipments_for_calc': (shipments_for_calc['id'] == 907).sum() == 0,
    'Проект 907 исключён из prolongation_base': (prolongation_base['id'] == 907).sum() == 0,
    'Нет пропусков в расчётных отгрузках': prolongation_base[
        ['end_month_shipment', 'month_1_shipment', 'month_2_shipment']
    ].isna().sum().sum() == 0,
    'Нет дублей в prolongation_base': base_duplicates == 0,
    'Нет конфликтов AM в prolongation_base': len(base_am_conflicts) == 0,
    'Нет нулевого знаменателя': (prolongation_base['end_month_shipment'] <= 0).sum() == 0,
    'Нет отрицательных отгрузок': (
        (prolongation_base['end_month_shipment'] < 0).sum()
        + (prolongation_base['month_1_shipment'] < 0).sum()
        + (prolongation_base['month_2_shipment'] < 0).sum()
    ) == 0
}

for check_name, result in checks.items():
    print(f'{check_name}:', 'OK' if result else 'ПРОВЕРИТЬ')

print('\nРазмеры ключевых таблиц:')
print('prolongations:', prolongations.shape)
print('prolongations_clean:', prolongations_clean.shape)
print('shipments_for_calc:', shipments_for_calc.shape)
print('prolongation_base:', prolongation_base.shape)

Нет дублей в prolongations: OK
Нет конфликтов AM в prolongations: OK
Проект 907 исключён из prolongations_clean: OK
Проект 907 исключён из shipments_for_calc: OK
Проект 907 исключён из prolongation_base: OK
Нет пропусков в расчётных отгрузках: OK
Нет дублей в prolongation_base: OK
Нет конфликтов AM в prolongation_base: OK
Нет нулевого знаменателя: OK
Нет отрицательных отгрузок: OK

Размеры ключевых таблиц:
prolongations: (473, 4)
prolongations_clean: (424, 4)
shipments_for_calc: (5008, 3)
prolongation_base: (398, 9)


## 8. Расчёт коэффициентов пролонгации

In [33]:
# Добавляем флаги пролонгации.
# 1-й месяц: есть положительная отгрузка в M+1.
# 2-й месяц: проект не был пролонгирован в M+1, но дал положительную отгрузку в M+2.

prolongation_base['is_prolonged_month_1'] = (
    prolongation_base['month_1_shipment'] > 0
)

prolongation_base['is_base_for_month_2'] = (
    prolongation_base['month_1_shipment'] == 0
)

prolongation_base['is_prolonged_month_2'] = (
    prolongation_base['is_base_for_month_2']
    & (prolongation_base['month_2_shipment'] > 0)
)

display(prolongation_base.head())

,id,month,AM,month_dt,month_1_dt,month_2_dt,end_month_shipment,month_1_shipment,month_2_shipment,is_prolonged_month_1,is_base_for_month_2,is_prolonged_month_2
0,42,ноябрь 2022,Васильев Артем Александрович,2022-11-01,2022-12-01,2023-01-01,36220.0,0.0,0.0,False,True,False
2,548,ноябрь 2022,Михайлов Андрей Сергеевич,2022-11-01,2022-12-01,2023-01-01,674000.0,674000.0,674000.0,True,False,False
3,87,ноябрь 2022,Соколова Анастасия Викторовна,2022-11-01,2022-12-01,2023-01-01,70050.0,0.0,73380.0,False,True,True
4,429,ноябрь 2022,Соколова Анастасия Викторовна,2022-11-01,2022-12-01,2023-01-01,30280.0,35580.0,35830.0,True,False,False
5,600,ноябрь 2022,Васильев Артем Александрович,2022-11-01,2022-12-01,2023-01-01,281417.0,175100.0,267220.0,True,False,False


In [34]:
# Формируем базы для коэффициентов.
# report_month — месяц, за который попадёт показатель в отчёт.

month_1_base = prolongation_base.copy()
month_1_base['report_month'] = month_1_base['month_1_dt']
month_1_base['month_1_numerator'] = np.where(
    month_1_base['is_prolonged_month_1'],
    month_1_base['month_1_shipment'],
    0
)
month_1_base['month_1_denominator'] = month_1_base['end_month_shipment']

month_2_base = prolongation_base[
    prolongation_base['is_base_for_month_2']
].copy()
month_2_base['report_month'] = month_2_base['month_2_dt']
month_2_base['month_2_numerator'] = np.where(
    month_2_base['is_prolonged_month_2'],
    month_2_base['month_2_shipment'],
    0
)
month_2_base['month_2_denominator'] = month_2_base['end_month_shipment']

report_start = pd.Timestamp('2023-01-01')
report_end = pd.Timestamp('2023-12-01')

month_1_base_2023 = month_1_base[
    (month_1_base['report_month'] >= report_start)
    & (month_1_base['report_month'] <= report_end)
].copy()

month_2_base_2023 = month_2_base[
    (month_2_base['report_month'] >= report_start)
    & (month_2_base['report_month'] <= report_end)
].copy()

print('Размер базы для коэффициента 1 месяца:', month_1_base_2023.shape)
print('Размер базы для коэффициента 2 месяца:', month_2_base_2023.shape)

Размер базы для коэффициента 1 месяца: (313, 15)
Размер базы для коэффициента 2 месяца: (152, 15)


In [35]:
# Функции расчёта коэффициентов.

def calculate_month_1_result(df, group_cols):
    result = (
        df
        .groupby(group_cols)
        .agg(
            month_1_numerator=('month_1_numerator', 'sum'),
            month_1_denominator=('month_1_denominator', 'sum'),
            projects_count_m1=('id', 'count'),
            prolonged_projects_m1=('is_prolonged_month_1', 'sum')
        )
        .reset_index()
    )
    result['coef_month_1'] = (
        result['month_1_numerator']
        / result['month_1_denominator']
    )
    return result

def calculate_month_2_result(df, group_cols):
    result = (
        df
        .groupby(group_cols)
        .agg(
            month_2_numerator=('month_2_numerator', 'sum'),
            month_2_denominator=('month_2_denominator', 'sum'),
            projects_count_m2=('id', 'count'),
            prolonged_projects_m2=('is_prolonged_month_2', 'sum')
        )
        .reset_index()
    )
    result['coef_month_2'] = (
        result['month_2_numerator']
        / result['month_2_denominator']
    )
    return result

In [36]:
# Коэффициенты по менеджерам и месяцам.

month_1_result = calculate_month_1_result(
    month_1_base_2023,
    ['AM', 'report_month']
)

month_2_result = calculate_month_2_result(
    month_2_base_2023,
    ['AM', 'report_month']
)

manager_month_result = (
    month_1_result
    .merge(month_2_result, on=['AM', 'report_month'], how='outer')
    .sort_values(['AM', 'report_month'])
)

display(manager_month_result.head(30))

,AM,report_month,month_1_numerator,month_1_denominator,projects_count_m1,prolonged_projects_m1,coef_month_1,month_2_numerator,month_2_denominator,projects_count_m2,prolonged_projects_m2,coef_month_2
0,Васильев Артем Александрович,2023-01-01,1012761.60,1693097.68,12.0,7.0,0.598171,0.00,260862.00,4.0,0.0,0.000000
1,Васильев Артем Александрович,2023-02-01,874015.00,827225.00,4.0,2.0,1.056563,44775.00,820335.00,5.0,1.0,0.054581
2,Васильев Артем Александрович,2023-03-01,621897.55,991467.42,9.0,5.0,0.627250,67774.08,95750.00,2.0,1.0,0.707823
3,Васильев Артем Александрович,2023-04-01,562360.00,1205775.45,11.0,5.0,0.466389,40200.00,364780.00,4.0,1.0,0.110203
4,Васильев Артем Александрович,2023-05-01,796824.00,2044918.50,9.0,5.0,0.389661,0.00,609590.90,6.0,0.0,0.000000
5,Васильев Артем Александрович,2023-06-01,151933.28,390286.00,4.0,2.0,0.389287,38632.50,678208.50,4.0,1.0,0.056963
6,Васильев Артем Александрович,2023-07-01,521691.00,891755.60,9.0,5.0,0.585016,0.00,230686.00,2.0,0.0,0.000000
7,Васильев Артем Александрович,2023-08-01,309900.00,649000.00,5.0,3.0,0.477504,0.00,492017.60,4.0,0.0,0.000000
8,Васильев Артем Александрович,2023-09-01,234774.00,1361540.74,8.0,3.0,0.172433,0.00,274500.00,2.0,0.0,0.000000
9,Васильев Артем Александрович,2023-10-01,355915.00,401370.00,6.0,1.0,0.886750,82020.00,1135125.00,5.0,1.0,0.072256


In [37]:
# Коэффициенты по всему отделу за каждый месяц.

department_month_1_result = calculate_month_1_result(
    month_1_base_2023,
    ['report_month']
)

department_month_2_result = calculate_month_2_result(
    month_2_base_2023,
    ['report_month']
)

department_month_result = (
    department_month_1_result
    .merge(department_month_2_result, on='report_month', how='outer')
    .sort_values('report_month')
)

display(department_month_result)

,report_month,month_1_numerator,month_1_denominator,projects_count_m1,prolonged_projects_m1,coef_month_1,month_2_numerator,month_2_denominator,projects_count_m2,prolonged_projects_m2,coef_month_2
0,2023-01-01,2748021.61,5986766.86,50,24,0.459016,73380.00,545022.00,9,1,0.134637
1,2023-02-01,2660945.00,3400075.50,17,7,0.782614,141120.00,2819042.76,26,3,0.050060
2,2023-03-01,1108904.40,1716452.77,20,12,0.646044,122999.08,760840.50,10,2,0.161662
3,2023-04-01,1501923.35,3762484.30,31,16,0.399184,57940.00,627735.00,8,2,0.092300
4,2023-05-01,1859461.75,3414631.50,22,16,0.544557,0.00,2181445.90,15,0,0.000000
5,2023-06-01,281673.28,1274625.53,19,3,0.220985,38632.50,1015068.50,6,1,0.038059
6,2023-07-01,1390411.00,2650982.85,28,14,0.524489,69385.00,1020325.53,16,1,0.068003
7,2023-08-01,943678.88,1936915.83,21,12,0.487207,52315.00,1477499.85,14,1,0.035408
8,2023-09-01,1086504.42,3113823.26,24,10,0.348929,0.00,755739.63,9,0,0.000000
9,2023-10-01,3014655.19,3686850.81,37,20,0.817678,82020.00,2054989.88,14,1,0.039913


In [38]:
# Годовые коэффициенты по менеджерам.
# Годовой коэффициент считается как сумма числителей / сумма знаменателей,
# а не как среднее месячных коэффициентов.

manager_year_1_result = (
    month_1_base_2023
    .groupby('AM')
    .agg(
        year_month_1_numerator=('month_1_numerator', 'sum'),
        year_month_1_denominator=('month_1_denominator', 'sum'),
        year_projects_count_m1=('id', 'count'),
        year_prolonged_projects_m1=('is_prolonged_month_1', 'sum')
    )
    .reset_index()
)

manager_year_1_result['year_coef_month_1'] = (
    manager_year_1_result['year_month_1_numerator']
    / manager_year_1_result['year_month_1_denominator']
)

manager_year_2_result = (
    month_2_base_2023
    .groupby('AM')
    .agg(
        year_month_2_numerator=('month_2_numerator', 'sum'),
        year_month_2_denominator=('month_2_denominator', 'sum'),
        year_projects_count_m2=('id', 'count'),
        year_prolonged_projects_m2=('is_prolonged_month_2', 'sum')
    )
    .reset_index()
)

manager_year_2_result['year_coef_month_2'] = (
    manager_year_2_result['year_month_2_numerator']
    / manager_year_2_result['year_month_2_denominator']
)

manager_year_result = (
    manager_year_1_result
    .merge(manager_year_2_result, on='AM', how='outer')
    .sort_values('AM')
)

display(manager_year_result)

,AM,year_month_1_numerator,year_month_1_denominator,year_projects_count_m1,year_prolonged_projects_m1,year_coef_month_1,year_month_2_numerator,year_month_2_denominator,year_projects_count_m2,year_prolonged_projects_m2,year_coef_month_2
0,Васильев Артем Александрович,5855965.43,11204084.89,85,42,0.522663,360941.58,5352587.50,45.0,6.0,0.067433
1,Иванова Мария Сергеевна,1660095.55,4727657.16,40,15,0.351146,0.00,2503864.75,26.0,0.0,0.000000
2,Кузнецов Михаил Иванович,470182.98,982724.44,11,7,0.478448,0.00,167675.00,2.0,0.0,0.000000
3,Михайлов Андрей Сергеевич,2235422.29,3361833.59,21,14,0.664941,0.00,1056004.68,7.0,0.0,0.000000
4,Петрова Анна Дмитриевна,109442.52,98492.00,1,1,1.111182,NaN,NaN,NaN,NaN,NaN
5,Попова Екатерина Николаевна,2502850.80,4890598.08,52,25,0.511768,39140.00,2363545.26,27.0,2.0,0.016560
6,Смирнова Ольга Владимировна,1925523.50,2817219.59,39,20,0.683484,203825.00,803500.80,14.0,4.0,0.253671
7,Соколова Анастасия Викторовна,3974267.97,6862934.34,64,34,0.579092,169725.00,2876881.74,31.0,3.0,0.058996


In [39]:
# Годовые коэффициенты по всему отделу.

department_year_result = pd.DataFrame({
    'period': ['2023'],

    'year_month_1_numerator': [month_1_base_2023['month_1_numerator'].sum()],
    'year_month_1_denominator': [month_1_base_2023['month_1_denominator'].sum()],
    'year_projects_count_m1': [month_1_base_2023['id'].count()],
    'year_prolonged_projects_m1': [month_1_base_2023['is_prolonged_month_1'].sum()],

    'year_month_2_numerator': [month_2_base_2023['month_2_numerator'].sum()],
    'year_month_2_denominator': [month_2_base_2023['month_2_denominator'].sum()],
    'year_projects_count_m2': [month_2_base_2023['id'].count()],
    'year_prolonged_projects_m2': [month_2_base_2023['is_prolonged_month_2'].sum()]
})

department_year_result['year_coef_month_1'] = (
    department_year_result['year_month_1_numerator']
    / department_year_result['year_month_1_denominator']
)

department_year_result['year_coef_month_2'] = (
    department_year_result['year_month_2_numerator']
    / department_year_result['year_month_2_denominator']
)

display(department_year_result)

,period,year_month_1_numerator,year_month_1_denominator,year_projects_count_m1,year_prolonged_projects_m1,year_month_2_numerator,year_month_2_denominator,year_projects_count_m2,year_prolonged_projects_m2,year_coef_month_1,year_coef_month_2
0,2023,18733751.04,34945544.09,313,158,773631.58,15124059.73,152,15,0.536084,0.051152


In [40]:
# Контроль расчётов коэффициентов:
# знаменатели должны быть положительными, коэффициенты не должны быть NaN там, где есть база.

print('month_1 denominator <= 0:',
      (month_1_result['month_1_denominator'] <= 0).sum())

print('month_2 denominator <= 0:',
      (month_2_result['month_2_denominator'] <= 0).sum())

print('NaN coef_month_1:',
      month_1_result['coef_month_1'].isna().sum())

print('NaN coef_month_2:',
      month_2_result['coef_month_2'].isna().sum())

month_1 denominator <= 0: 0
month_2 denominator <= 0: 0
NaN coef_month_1: 0
NaN coef_month_2: 0


## 9. Дополнительная проверка низкого коэффициента 2-го месяца

In [41]:
# Проверяем, почему годовой коэффициент 2-го месяца значительно ниже коэффициента 1-го месяца.
# Это диагностический блок: он подтверждает, что низкий показатель объясняется структурой данных,
# а не ошибкой расчёта.

second_month_success = month_2_base_2023[
    month_2_base_2023['is_prolonged_month_2']
].copy()

second_month_failed = month_2_base_2023[
    ~month_2_base_2023['is_prolonged_month_2']
].copy()

print('Количество всех попыток во 2 месяц:', len(month_2_base_2023))
print('Количество успешных попыток во 2 месяц:', month_2_base_2023['is_prolonged_month_2'].sum())
print('Количество неуспешных попыток во 2 месяц:', len(second_month_failed))

print('\nСумма базы всех попыток 2 месяца:',
      month_2_base_2023['end_month_shipment'].sum())

print('Сумма базы успешных попыток 2 месяца:',
      second_month_success['end_month_shipment'].sum())

print('Сумма базы неуспешных попыток 2 месяца:',
      second_month_failed['end_month_shipment'].sum())

display(
    second_month_failed[
        ['id', 'AM', 'month', 'month_dt',
         'end_month_shipment', 'month_1_shipment', 'month_2_shipment']
    ]
    .sort_values('end_month_shipment', ascending=False)
    .head(20)
)

Количество всех попыток во 2 месяц: 152
Количество успешных попыток во 2 месяц: 15
Количество неуспешных попыток во 2 месяц: 137

Сумма базы всех попыток 2 месяца: 15124059.73
Сумма базы успешных попыток 2 месяца: 723059.35
Сумма базы неуспешных попыток 2 месяца: 14401000.379999999


,id,AM,month,month_dt,end_month_shipment,month_1_shipment,month_2_shipment
127,728,Попова Екатерина Николаевна,март 2023,2023-03-01,780000.00,0.0,0.0
264,916,Васильев Артем Александрович,август 2023,2023-08-01,763110.00,0.0,0.0
30,719,Васильев Артем Александрович,декабрь 2022,2022-12-01,602370.00,0.0,0.0
174,838,Васильев Артем Александрович,апрель 2023,2023-04-01,550000.00,0.0,0.0
263,781,Михайлов Андрей Сергеевич,август 2023,2023-08-01,333853.75,0.0,0.0
218,857,Соколова Анастасия Викторовна,июнь 2023,2023-06-01,321280.00,0.0,0.0
306,782,Михайлов Андрей Сергеевич,сентябрь 2023,2023-09-01,316620.75,0.0,0.0
141,682,Васильев Артем Александрович,март 2023,2023-03-01,276007.90,0.0,0.0
207,600,Васильев Артем Александрович,июнь 2023,2023-06-01,257800.00,0.0,0.0
134,506,Иванова Мария Сергеевна,март 2023,2023-03-01,257700.00,0.0,0.0


## 10. Подготовка таблиц для отчёта и выгрузка CSV

In [42]:
# Служебные функции для подготовки отчётных таблиц.

month_names_ru = {
    1: 'Январь',
    2: 'Февраль',
    3: 'Март',
    4: 'Апрель',
    5: 'Май',
    6: 'Июнь',
    7: 'Июль',
    8: 'Август',
    9: 'Сентябрь',
    10: 'Октябрь',
    11: 'Ноябрь',
    12: 'Декабрь'
}

months_order = [
    'Январь', 'Февраль', 'Март', 'Апрель', 'Май', 'Июнь',
    'Июль', 'Август', 'Сентябрь', 'Октябрь', 'Ноябрь', 'Декабрь'
]

def format_month_ru(dt):
    return month_names_ru[dt.month]

def round_money(value):
    if pd.isna(value):
        return np.nan
    return round(value, 2)

def round_coef(value):
    if pd.isna(value):
        return np.nan
    return round(value, 4)

def coef_to_percent_text(value):
    if pd.isna(value):
        return ''
    return f'{value:.2%}'

In [43]:
# Таблица для листа "Весь отдел".

report_department_month = department_month_result.copy()
report_department_month['Месяц'] = report_department_month['report_month'].apply(format_month_ru)

report_department_month = report_department_month[
    [
        'Месяц',
        'month_1_denominator',
        'month_1_numerator',
        'coef_month_1',
        'month_2_denominator',
        'month_2_numerator',
        'coef_month_2'
    ]
].rename(columns={
    'month_1_denominator': 'К пролонгации 1 месяц',
    'month_1_numerator': 'Пролонгировано 1 месяц',
    'coef_month_1': 'Коэффициент 1 месяц',
    'month_2_denominator': 'К пролонгации 2 месяц',
    'month_2_numerator': 'Пролонгировано 2 месяц',
    'coef_month_2': 'Коэффициент 2 месяц'
})

money_cols = [
    'К пролонгации 1 месяц',
    'Пролонгировано 1 месяц',
    'К пролонгации 2 месяц',
    'Пролонгировано 2 месяц'
]

coef_cols = [
    'Коэффициент 1 месяц',
    'Коэффициент 2 месяц'
]

for col in money_cols:
    report_department_month[col] = report_department_month[col].apply(round_money)

for col in coef_cols:
    report_department_month[col] = report_department_month[col].apply(round_coef)

display(report_department_month)

,Месяц,К пролонгации 1 месяц,Пролонгировано 1 месяц,Коэффициент 1 месяц,К пролонгации 2 месяц,Пролонгировано 2 месяц,Коэффициент 2 месяц
0,Январь,5986766.86,2748021.61,0.4590,545022.00,73380.00,0.1346
1,Февраль,3400075.50,2660945.00,0.7826,2819042.76,141120.00,0.0501
2,Март,1716452.77,1108904.40,0.6460,760840.50,122999.08,0.1617
3,Апрель,3762484.30,1501923.35,0.3992,627735.00,57940.00,0.0923
4,Май,3414631.50,1859461.75,0.5446,2181445.90,0.00,0.0000
5,Июнь,1274625.53,281673.28,0.2210,1015068.50,38632.50,0.0381
6,Июль,2650982.85,1390411.00,0.5245,1020325.53,69385.00,0.0680
7,Август,1936915.83,943678.88,0.4872,1477499.85,52315.00,0.0354
8,Сентябрь,3113823.26,1086504.42,0.3489,755739.63,0.00,0.0000
9,Октябрь,3686850.81,3014655.19,0.8177,2054989.88,82020.00,0.0399


In [44]:
# Таблица для листа "Менеджеры за год".

report_manager_year = manager_year_result.copy()

report_manager_year = report_manager_year[
    [
        'AM',
        'year_month_1_denominator',
        'year_month_1_numerator',
        'year_coef_month_1',
        'year_month_2_denominator',
        'year_month_2_numerator',
        'year_coef_month_2'
    ]
].rename(columns={
    'AM': 'Менеджер',
    'year_month_1_denominator': 'К пролонгации 1 месяц',
    'year_month_1_numerator': 'Пролонгировано 1 месяц',
    'year_coef_month_1': 'Коэффициент 1 месяц',
    'year_month_2_denominator': 'К пролонгации 2 месяц',
    'year_month_2_numerator': 'Пролонгировано 2 месяц',
    'year_coef_month_2': 'Коэффициент 2 месяц'
})

for col in money_cols:
    report_manager_year[col] = report_manager_year[col].apply(round_money)

for col in coef_cols:
    report_manager_year[col] = report_manager_year[col].apply(round_coef)

display(report_manager_year)

,Менеджер,К пролонгации 1 месяц,Пролонгировано 1 месяц,Коэффициент 1 месяц,К пролонгации 2 месяц,Пролонгировано 2 месяц,Коэффициент 2 месяц
0,Васильев Артем Александрович,11204084.89,5855965.43,0.5227,5352587.50,360941.58,0.0674
1,Иванова Мария Сергеевна,4727657.16,1660095.55,0.3511,2503864.75,0.00,0.0000
2,Кузнецов Михаил Иванович,982724.44,470182.98,0.4784,167675.00,0.00,0.0000
3,Михайлов Андрей Сергеевич,3361833.59,2235422.29,0.6649,1056004.68,0.00,0.0000
4,Петрова Анна Дмитриевна,98492.00,109442.52,1.1112,NaN,NaN,NaN
5,Попова Екатерина Николаевна,4890598.08,2502850.80,0.5118,2363545.26,39140.00,0.0166
6,Смирнова Ольга Владимировна,2817219.59,1925523.50,0.6835,803500.80,203825.00,0.2537
7,Соколова Анастасия Викторовна,6862934.34,3974267.97,0.5791,2876881.74,169725.00,0.0590


In [45]:
# Таблица "Менеджеры по месяцам": коэффициент 1.
# Значения оставляем числовыми; отдельная export-версия ниже сохранит проценты как текст.

manager_month_coef_1 = manager_month_result.copy()
manager_month_coef_1['Месяц'] = manager_month_coef_1['report_month'].apply(format_month_ru)

report_manager_month_coef_1 = (
    manager_month_coef_1
    .pivot_table(
        index='AM',
        columns='Месяц',
        values='coef_month_1',
        aggfunc='first'
    )
    .reset_index()
    .rename(columns={'AM': 'Менеджер'})
)

existing_months_1 = [
    month for month in months_order
    if month in report_manager_month_coef_1.columns
]

report_manager_month_coef_1 = report_manager_month_coef_1[
    ['Менеджер'] + existing_months_1
]

for col in existing_months_1:
    report_manager_month_coef_1[col] = report_manager_month_coef_1[col].apply(round_coef)

display(report_manager_month_coef_1)

Месяц,Менеджер,Январь,Февраль,Март,Апрель,Май,Июнь,Июль,Август,Сентябрь,Октябрь,Ноябрь,Декабрь
0,Васильев Артем Александрович,0.5982,1.0566,0.6272,0.4664,0.3897,0.3893,0.5850,0.4775,0.1724,0.8868,0.6210,0.3241
1,Иванова Мария Сергеевна,0.2701,NaN,0.4461,0.2777,1.0000,0.0000,0.4726,0.5407,0.9985,NaN,NaN,NaN
2,Кузнецов Михаил Иванович,1.2660,0.8470,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,1.3085,0.4736,0.2740
3,Михайлов Андрей Сергеевич,0.6879,0.9036,2.4306,1.0988,1.1896,0.0000,1.1975,NaN,0.0000,0.4491,NaN,0.0000
4,Петрова Анна Дмитриевна,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.1112
5,Попова Екатерина Николаевна,0.4407,0.8009,0.4963,0.0954,0.7569,0.0000,0.0891,0.4799,0.7470,0.7936,NaN,0.6001
6,Смирнова Ольга Владимировна,0.7434,0.0000,NaN,0.9661,NaN,0.0000,0.0000,0.3725,0.7655,1.0384,0.8362,0.3611
7,Соколова Анастасия Викторовна,0.4273,0.2347,0.9392,1.0658,0.2374,0.7664,0.6191,0.5413,0.2602,0.8418,0.1786,0.7911


In [46]:
# Таблица "Менеджеры по месяцам": коэффициент 2.

manager_month_coef_2 = manager_month_result.copy()
manager_month_coef_2['Месяц'] = manager_month_coef_2['report_month'].apply(format_month_ru)

report_manager_month_coef_2 = (
    manager_month_coef_2
    .pivot_table(
        index='AM',
        columns='Месяц',
        values='coef_month_2',
        aggfunc='first'
    )
    .reset_index()
    .rename(columns={'AM': 'Менеджер'})
)

existing_months_2 = [
    month for month in months_order
    if month in report_manager_month_coef_2.columns
]

report_manager_month_coef_2 = report_manager_month_coef_2[
    ['Менеджер'] + existing_months_2
]

for col in existing_months_2:
    report_manager_month_coef_2[col] = report_manager_month_coef_2[col].apply(round_coef)

display(report_manager_month_coef_2)

Месяц,Менеджер,Январь,Февраль,Март,Апрель,Май,Июнь,Июль,Август,Сентябрь,Октябрь,Ноябрь,Декабрь
0,Васильев Артем Александрович,0.0000,0.0546,0.7078,0.1102,0.0,0.057,0.0000,0.0000,0.0,0.0723,0.0000,0.4849
1,Иванова Мария Сергеевна,0.0000,0.0000,NaN,0.0000,0.0,NaN,0.0000,0.0000,0.0,NaN,NaN,NaN
2,Кузнецов Михаил Иванович,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0000
3,Михайлов Андрей Сергеевич,0.0000,0.0000,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,0.0000,0.0000,NaN
4,Попова Екатерина Николаевна,0.0000,0.0000,0.0000,0.1909,0.0,0.000,0.0000,0.0000,0.0,0.0000,0.1898,NaN
5,Смирнова Ольга Владимировна,NaN,0.0000,1.1505,NaN,NaN,NaN,0.5163,1.5531,0.0,0.0000,0.0000,0.3984
6,Соколова Анастасия Викторовна,0.6788,0.2157,0.0000,0.0000,0.0,0.000,0.0000,0.0000,0.0,0.0000,0.0000,0.0000


In [47]:
# Таблица для диаграммы "Менеджеры за год":
# рейтинг по коэффициенту 1-го месяца + база для контекста.

chart_manager_year_coef_1 = (
    report_manager_year[
        ['Менеджер', 'Коэффициент 1 месяц', 'К пролонгации 1 месяц']
    ]
    .sort_values('Коэффициент 1 месяц', ascending=False)
    .copy()
)

display(chart_manager_year_coef_1)

,Менеджер,Коэффициент 1 месяц,К пролонгации 1 месяц
4,Петрова Анна Дмитриевна,1.1112,98492.00
6,Смирнова Ольга Владимировна,0.6835,2817219.59
3,Михайлов Андрей Сергеевич,0.6649,3361833.59
7,Соколова Анастасия Викторовна,0.5791,6862934.34
0,Васильев Артем Александрович,0.5227,11204084.89
5,Попова Екатерина Николаевна,0.5118,4890598.08
2,Кузнецов Михаил Иванович,0.4784,982724.44
1,Иванова Мария Сергеевна,0.3511,4727657.16


In [48]:
# Для Google Sheets готовим версии помесячных таблиц,
# где коэффициенты уже записаны как процентный текст.
# Это помогает избежать ошибок форматирования при импорте.

report_manager_month_coef_1_percent = report_manager_month_coef_1.copy()

for col in report_manager_month_coef_1_percent.columns:
    if col != 'Менеджер':
        report_manager_month_coef_1_percent[col] = (
            report_manager_month_coef_1_percent[col].apply(coef_to_percent_text)
        )

report_manager_month_coef_2_percent = report_manager_month_coef_2.copy()

for col in report_manager_month_coef_2_percent.columns:
    if col != 'Менеджер':
        report_manager_month_coef_2_percent[col] = (
            report_manager_month_coef_2_percent[col].apply(coef_to_percent_text)
        )

display(report_manager_month_coef_1_percent)
display(report_manager_month_coef_2_percent)

Месяц,Менеджер,Январь,Февраль,Март,Апрель,Май,Июнь,Июль,Август,Сентябрь,Октябрь,Ноябрь,Декабрь
0,Васильев Артем Александрович,59.82%,105.66%,62.72%,46.64%,38.97%,38.93%,58.50%,47.75%,17.24%,88.68%,62.10%,32.41%
1,Иванова Мария Сергеевна,27.01%,,44.61%,27.77%,100.00%,0.00%,47.26%,54.07%,99.85%,,,
2,Кузнецов Михаил Иванович,126.60%,84.70%,,0.00%,,,,,,130.85%,47.36%,27.40%
3,Михайлов Андрей Сергеевич,68.79%,90.36%,243.06%,109.88%,118.96%,0.00%,119.75%,,0.00%,44.91%,,0.00%
4,Петрова Анна Дмитриевна,,,,,,,,,,,,111.12%
5,Попова Екатерина Николаевна,44.07%,80.09%,49.63%,9.54%,75.69%,0.00%,8.91%,47.99%,74.70%,79.36%,,60.01%
6,Смирнова Ольга Владимировна,74.34%,0.00%,,96.61%,,0.00%,0.00%,37.25%,76.55%,103.84%,83.62%,36.11%
7,Соколова Анастасия Викторовна,42.73%,23.47%,93.92%,106.58%,23.74%,76.64%,61.91%,54.13%,26.02%,84.18%,17.86%,79.11%


Месяц,Менеджер,Январь,Февраль,Март,Апрель,Май,Июнь,Июль,Август,Сентябрь,Октябрь,Ноябрь,Декабрь
0,Васильев Артем Александрович,0.00%,5.46%,70.78%,11.02%,0.00%,5.70%,0.00%,0.00%,0.00%,7.23%,0.00%,48.49%
1,Иванова Мария Сергеевна,0.00%,0.00%,,0.00%,0.00%,,0.00%,0.00%,0.00%,,,
2,Кузнецов Михаил Иванович,,,,,0.00%,,,,,,,0.00%
3,Михайлов Андрей Сергеевич,0.00%,0.00%,,,,,0.00%,,,0.00%,0.00%,
4,Попова Екатерина Николаевна,0.00%,0.00%,0.00%,19.09%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,18.98%,
5,Смирнова Ольга Владимировна,,0.00%,115.05%,,,,51.63%,155.31%,0.00%,0.00%,0.00%,39.84%
6,Соколова Анастасия Викторовна,67.88%,21.57%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%


In [49]:
# Сохраняем таблицы в CSV для загрузки в Google Sheets.
# encoding='utf-8-sig' помогает корректно открыть русские символы в Excel/Google Sheets.

report_department_month.to_csv(
    '01_ves_otdel.csv',
    index=False,
    encoding='utf-8-sig'
)

report_manager_year.to_csv(
    '02_menedzhery_za_god.csv',
    index=False,
    encoding='utf-8-sig',
    na_rep=''
)

report_manager_month_coef_1_percent.to_csv(
    '03_menedzhery_po_mesyacam_koef_1_percent.csv',
    index=False,
    encoding='utf-8-sig'
)

report_manager_month_coef_2_percent.to_csv(
    '04_menedzhery_po_mesyacam_koef_2_percent.csv',
    index=False,
    encoding='utf-8-sig'
)

chart_manager_year_coef_1.to_csv(
    '05_chart_manager_year_coef_1.csv',
    index=False,
    encoding='utf-8-sig',
    na_rep=''
)

print('CSV-файлы сохранены.')

CSV-файлы сохранены.

## 11. Краткий аналитический вывод

- По итогам 2023 года основной объём пролонгаций формируется в первый месяц после завершения проекта.
- Годовой коэффициент пролонгации отдела в первый месяц составил около **53,6%**.
- Коэффициент пролонгации через месяц заметно ниже — около **5,1%**. Это связано с тем, что во второй месяц попадают только проекты, которые не были пролонгированы сразу.
- Низкий коэффициент 2-го месяца подтверждён дополнительной проверкой: большая часть базы второго месяца относится к проектам, которые не вернулись ни в первый, ни во второй месяц.
- Значения коэффициентов выше 100% не являются ошибкой: коэффициент считается по сумме отгрузки, и после пролонгации сумма может быть выше, чем в последний месяц реализации.